11- Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.


In [0]:
%sql
SELECT
  COUNT(*)       AS reservas_canceladas_agencia,
  COUNT(*) * 1.5 AS perdida_comision_usd
FROM cursodataacademy.hotel_db.tb_reservas
WHERE distribution_channel = 'TA/TO'
  AND is_canceled = 1;

Conclusión:Se pagaron USD 60,228 en comisiones por 40,152 reservas de agencia que terminaron canceladas. Es dinero sin retorno, así que conviene renegociar el pago para que se cobre solo cuando la reserva se concrete.

12- El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
WITH fechas AS (
  SELECT
    is_canceled,
    TO_DATE(CONCAT(arrival_date_year, ' ', arrival_date_month, ' ', arrival_date_day_of_month), 'yyyy MMMM d') AS arrival_date,
    TO_DATE(reservation_status_date, 'd/M/yy') AS reservation_status_date
  FROM cursodataacademy.hotel_db.tb_reservas
)
SELECT
  COUNT(*)       AS cancelaciones_tardias,
  COUNT(*) * 120 AS perdida_total_usd
FROM fechas
WHERE is_canceled = 1
  AND DATEDIFF(arrival_date, reservation_status_date) < 3;

Conclusión: 3,204 reservas se cancelaron con menos de 3 días de anticipación, lo que genera una pérdida de USD 384,480. Se justifica aplicar penalidades o tarifas no reembolsables para las cancelaciones de último momento.

- Creacion de Tablas para los siguientes ejercicios 

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_paises (
  cod_pais STRING,
  nombre_pais STRING
);

INSERT INTO cursodataacademy.hotel_db.tb_paises VALUES
  ('PRT','Portugal'),('ESP','España'),('GBR','Reino Unido'),('FRA','Francia'),
  ('DEU','Alemania'),('ITA','Italia'),('IRL','Irlanda'),('BEL','Bélgica'),
  ('NLD','Países Bajos'),('CHE','Suiza'),('USA','Estados Unidos'),('BRA','Brasil'),
  ('AUT','Austria'),('SWE','Suecia'),('CHN','China'),('POL','Polonia'),
  ('PER','Perú'),('MEX','México'),('ARG','Argentina'),('CHL','Chile');

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_vendedores (
  id_vendedor INT,
  nombre_vendedor STRING
);

INSERT INTO cursodataacademy.hotel_db.tb_vendedores VALUES
  (9,'Carlos Ruiz'),(240,'Ana Torres'),(1,'Luis Pérez'),
  (14,'María Gómez'),(7,'Jorge Díaz'),(900,'Pedro Salas'),(901,'Lucía Vega');

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_reservas_id AS
SELECT ROW_NUMBER() OVER (ORDER BY hotel) AS id_reserva, *
FROM cursodataacademy.hotel_db.tb_reservas;

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_facturas (
  id_reserva INT,
  monto_facturado DECIMAL(10,2)
);

INSERT INTO cursodataacademy.hotel_db.tb_facturas VALUES
  (1,350.00),(2,420.50),(3,180.00),(4,275.30),(5,510.00);

In [0]:
%sql
SHOW TABLES IN cursodataacademy.hotel_db;

13- Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.


In [0]:
%sql
SELECT
  tb_reservas.is_canceled,
  tb_paises.nombre_pais
FROM cursodataacademy.hotel_db.tb_reservas
LEFT JOIN cursodataacademy.hotel_db.tb_paises
  ON tb_reservas.country = tb_paises.cod_pais;

Conclusión: El LEFT JOIN mantiene todas las reservas de tb_reservas, incluso las de países que no están registrados en tb_paises, que aparecen con nombre_pais en NULL. Esto permite ver si cada reserva fue cancelada y de qué país viene, pero muestra que hay que completar el catálogo de países para no perder información en el análisis por origen.

14- Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?


In [0]:
%sql
SELECT
  tb_paises.nombre_pais,
  COUNT(*) AS total_canceladas
FROM cursodataacademy.hotel_db.tb_reservas
INNER JOIN cursodataacademy.hotel_db.tb_paises
  ON tb_reservas.country = tb_paises.cod_pais
WHERE tb_reservas.is_canceled = 1
GROUP BY tb_paises.nombre_pais
ORDER BY total_canceladas DESC
LIMIT 5;

Conclusión: Portugal lidera con 27,519 cancelaciones, muy por encima de Reino Unido (2,453), España, Francia e Italia. Más de la mitad de las cancelaciones vienen de un solo mercado, así que las políticas de reducción deben enfocarse ahí primero.

15- Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.


In [0]:
%sql
SELECT
  tb_reservas.distribution_channel,
  tb_vendedores.nombre_vendedor
FROM cursodataacademy.hotel_db.tb_reservas
LEFT JOIN cursodataacademy.hotel_db.tb_vendedores
  ON CAST(tb_reservas.agent AS STRING) = CAST(tb_vendedores.id_vendedor AS STRING)

UNION

SELECT
  tb_reservas.distribution_channel,
  tb_vendedores.nombre_vendedor
FROM cursodataacademy.hotel_db.tb_reservas
RIGHT JOIN cursodataacademy.hotel_db.tb_vendedores
  ON CAST(tb_reservas.agent AS STRING) = CAST(tb_vendedores.id_vendedor AS STRING);

Conclusión: Hay reservas sin vendedor asignado (nombre en NULL) y vendedores sin ninguna reserva (canal en NULL), como Pedro Salas y Lucía Vega. Falta control en la asignación de reservas, lo que afecta el cálculo de comisiones y el seguimiento del equipo de ventas.

16- El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).


In [0]:
%sql
SELECT
  tb_facturas.id_reserva,
  tb_facturas.monto_facturado,
  tb_reservas_id.hotel
FROM cursodataacademy.hotel_db.tb_reservas_id
RIGHT JOIN cursodataacademy.hotel_db.tb_facturas
  ON tb_reservas_id.id_reserva = tb_facturas.id_reserva;

Conclusión: El RIGHT JOIN muestra todas las facturas emitidas (5 en total), y cada una se cruzó con su reserva. Las 5 corresponden a City Hotel, así que contabilidad puede ver a qué hotel pertenece cada monto facturado, que suma USD 1,735.80.